# Tarea 23 · Escalado e integración del preprocesamiento

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/julianschmidt1/tpi-analisis-de-datos/blob/main/notebooks/trabajo/23_escalado.ipynb)

**TPI · Introducción al Análisis de Datos · UTN FRA · División 343**  
**Integrantes:** Matías, Julián y Emmanuel. **Responsable:** Julián.

Integramos la codificación de la tarea 22 y el escalado en un único `preprocessor`. Entrada: los cuatro CSV originales de `data/processed/particion/` (tarea 21). Salida: cuatro CSV finales en `data/processed/`, listos para clasificación.

## Cómo ejecutarlo

- **Local:** instalar `requirements.txt` y ejecutar todas las celdas desde una carpeta del repositorio.
- **Colab:** subir este notebook y los cuatro archivos de la tarea 21 a `data/processed/particion/`. El badge estará disponible al publicar el notebook en `main`. No volver a partir los datos ni usar los CSV ya codificados de la tarea 22.
- Las salidas quedan en `data/processed/`; en Colab se descargan desde el panel **Archivos**.


## 1. Carga y validación de la partición

Leemos `fila_origen` como índice, nunca como predictor. Conservamos los clientes y las etiquetas de la tarea 21. Comprobamos alineación, ausencia de nulos y separación entre train y test antes de ajustar transformaciones.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler

try:
    import google.colab
except ImportError:
    EN_COLAB = False
else:
    EN_COLAB = True

actual = Path.cwd().resolve()
RAIZ = actual if EN_COLAB else next(
    (p for p in (actual, *actual.parents) if (p / 'PLAN_TAREAS.md').is_file()), None)
if RAIZ is None:
    raise FileNotFoundError('Abrí Jupyter dentro del repositorio del TPI.')
entrada = RAIZ / 'data' / 'processed' / 'particion'
archivos = ['X_train', 'X_test', 'y_train', 'y_test']
faltantes = [n for n in archivos if not (entrada / f'{n}.csv').is_file()]
if faltantes:
    raise FileNotFoundError(f'Faltan {faltantes} en {entrada}. Ejecutá la tarea 21 o copiá sus salidas.')
X_train = pd.read_csv(entrada / 'X_train.csv', index_col='fila_origen')
X_test = pd.read_csv(entrada / 'X_test.csv', index_col='fila_origen')
y_train = pd.read_csv(entrada / 'y_train.csv', index_col='fila_origen')['Churn']
y_test = pd.read_csv(entrada / 'y_test.csv', index_col='fila_origen')['Churn']
assert X_train.index.is_unique and X_test.index.is_unique
assert X_train.index.intersection(X_test.index).empty
assert X_train.index.equals(y_train.index) and X_test.index.equals(y_test.index)
assert X_train.columns.equals(X_test.columns)
assert 'Churn' not in X_train.columns and 'fila_origen' not in X_train.columns
assert set(y_train.unique()) == set(y_test.unique()) == {0, 1}
assert not X_train.isna().any().any() and not X_test.isna().any().any()
print(f'Train: {X_train.shape}; test: {X_test.shape}. Etiquetas alineadas.')

Train: (5634, 21); test: (1409, 21). Etiquetas alineadas.


## 2. Esquema integrado y decisiones

Reutilizamos One-Hot completo para las nominales, incluido `PaymentMethod`, y el orden contractual mensual=0, un año=1, dos años=2. Las categorías nominales se aprenden solo de train; las desconocidas se ignoran. Un contrato desconocido produce un error para revisar el dato.

Escalamos `tenure`, `MonthlyCharges`, `TotalCharges` y las columnas calculadas `ingreso_estimado` y `diferencia_facturacion`: dejar estas últimas sin escalar mantendría diferencias grandes de magnitud. No escalamos los indicadores One-Hot ni el contrato ordinal. El escalado no elimina la redundancia entre variables calculadas y originales; habrá que considerarla al interpretar coeficientes.

`StandardScaler` centra y divide por el desvío poblacional de train (`ddof=0`). `MinMaxScaler` usa los extremos de train para llevar ese conjunto a [0, 1]. Comparamos sus escalas sin entrenar modelos ni elegir según test. Adoptamos StandardScaler por el centrado y las unidades de desvío, útiles para la regresión logística prevista; no afirmamos que tenga mejor rendimiento predictivo.

In [2]:
orden_contrato = ['Month-to-month', 'One year', 'Two year']
columnas_nominales = X_train.select_dtypes(include=['object', 'string', 'category']).columns.drop('Contract').tolist()
columnas_numericas = X_train.select_dtypes(include='number').columns.tolist()
assert set(columnas_numericas) == {'tenure', 'MonthlyCharges', 'TotalCharges', 'ingreso_estimado', 'diferencia_facturacion'}
assert set(columnas_nominales + ['Contract'] + columnas_numericas) == set(X_train.columns)

def crear_preprocessor(escalador):
    return ColumnTransformer([
        ('nominal', OneHotEncoder(handle_unknown='ignore', drop=None, sparse_output=False), columnas_nominales),
        ('contrato', OrdinalEncoder(categories=[orden_contrato], handle_unknown='error'), ['Contract']),
        ('numericas', escalador, columnas_numericas),
    ], remainder='drop', verbose_feature_names_out=True)

preprocessor = crear_preprocessor(StandardScaler())
preprocessor_minmax = crear_preprocessor(MinMaxScaler())
train_standard = preprocessor.fit_transform(X_train)
train_minmax = preprocessor_minmax.fit_transform(X_train)
resumen = pd.DataFrame({
    'media_original': X_train[columnas_numericas].mean(),
    'desvio_original': X_train[columnas_numericas].std(ddof=0),
    'media_standard': train_standard[:, preprocessor.output_indices_['numericas']].mean(axis=0),
    'desvio_standard': train_standard[:, preprocessor.output_indices_['numericas']].std(axis=0),
    'min_minmax': train_minmax[:, preprocessor_minmax.output_indices_['numericas']].min(axis=0),
    'max_minmax': train_minmax[:, preprocessor_minmax.output_indices_['numericas']].max(axis=0),
}, index=columnas_numericas)
display(resumen.round(6))

,media_original,desvio_original,media_standard,desvio_standard,min_minmax,max_minmax
tenure,32.485091,24.566563,-0.0,1.0,0.0,1.0
MonthlyCharges,64.929961,30.135431,-0.0,1.0,0.0,1.0
TotalCharges,2299.334682,2279.001996,-0.0,1.0,0.0,1.0
ingreso_estimado,2299.296645,2277.628473,0.0,1.0,0.0,1.0
diferencia_facturacion,0.038037,67.440702,0.0,1.0,0.0,1.0


La tabla contrasta las unidades originales con ambas transformaciones. StandardScaler produce media cercana a cero y desvío cercano a uno en entrenamiento; MinMaxScaler lleva sus extremos a cero y uno. Ninguno cambia el orden de los clientes ni elimina valores extremos. En datos nuevos, StandardScaler no garantiza media cero ni desvío uno, y MinMaxScaler puede producir valores fuera de [0, 1].

## 3. Transformación de test y matrices finales

Transformamos test únicamente con el preprocesador StandardScaler ya ajustado. Conservamos los nombres de las variables y los índices para enlazar cada fila con su etiqueta. El objeto `preprocessor` queda disponible en memoria para la integración del notebook final.

In [3]:
test_standard = preprocessor.transform(X_test)
nombres = preprocessor.get_feature_names_out()
X_train_transformado = pd.DataFrame(train_standard, columns=nombres, index=X_train.index)
X_test_transformado = pd.DataFrame(test_standard, columns=nombres, index=X_test.index)
print(f'Train transformado: {X_train_transformado.shape}; test: {X_test_transformado.shape}')
display(X_train_transformado.head())

Train transformado: (5634, 46); test: (1409, 46)


,nominal__gender_Female,nominal__gender_Male,nominal__SeniorCitizen_No,nominal__SeniorCitizen_Yes,nominal__Partner_No,nominal__Partner_Yes,nominal__Dependents_No,nominal__Dependents_Yes,nominal__PhoneService_No,nominal__PhoneService_Yes,...,nominal__PaymentMethod_Bank transfer (automatic),nominal__PaymentMethod_Credit card (automatic),nominal__PaymentMethod_Electronic check,nominal__PaymentMethod_Mailed check,contrato__Contract,numericas__tenure,numericas__MonthlyCharges,numericas__TotalCharges,numericas__ingreso_estimado,numericas__diferencia_facturacion
fila_origen,,,,,,,,,,,,,,,,,,,,,
3738,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.102371,-0.521976,-0.262257,-0.253464,-0.302311
3151,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,-0.711743,0.337478,-0.503635,-0.514920,0.370873
4860,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,1.0,2.0,-0.793155,-0.809013,-0.749883,-0.778067,0.936556
3867,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,2.0,-0.263980,0.284384,-0.172722,-0.170483,-0.079152
3810,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,-1.281624,-0.676279,-0.989374,-0.989954,-0.000564


Los indicadores nominales siguen siendo binarios y el contrato conserva su orden. Las cinco variables numéricas ahora expresan distancias respecto de la media de entrenamiento en unidades de su desvío. Las matrices quedan preparadas para entrenar el clasificador, manteniendo test reservado para evaluación.

## 4. Verificación del ajuste y de las salidas

Contrastamos las estadísticas aprendidas con train y la transformación de test con la fórmula manual. Comprobamos también categorías, contrato, finitud y alineación de etiquetas. Esta comprobación cubre el preprocesamiento de esta tarea; la revisión transversal corresponde a la tarea 24.

In [4]:
scaler = preprocessor.named_transformers_['numericas']
np.testing.assert_allclose(scaler.mean_, X_train[columnas_numericas].mean())
np.testing.assert_allclose(scaler.var_, X_train[columnas_numericas].var(ddof=0))
assert scaler.n_samples_seen_ == len(X_train)
numericas_train = X_train_transformado.filter(like='numericas__')
np.testing.assert_allclose(numericas_train.mean(), 0, atol=1e-12)
np.testing.assert_allclose(numericas_train.std(ddof=0), 1, atol=1e-12)
np.testing.assert_allclose(
    X_test_transformado.filter(like='numericas__'),
    (X_test[columnas_numericas].to_numpy() - scaler.mean_) / scaler.scale_)
np.testing.assert_allclose(resumen['min_minmax'], 0, atol=1e-12)
np.testing.assert_allclose(resumen['max_minmax'], 1, atol=1e-12)
for columna, categorias in zip(columnas_nominales, preprocessor.named_transformers_['nominal'].categories_):
    assert set(categorias) == set(X_train[columna].unique())
for original, convertido, objetivo in [(X_train, X_train_transformado, y_train), (X_test, X_test_transformado, y_test)]:
    assert convertido.index.equals(objetivo.index)
    assert convertido.columns.is_unique
    assert all(pd.api.types.is_numeric_dtype(t) for t in convertido.dtypes)
    assert np.isfinite(convertido.to_numpy()).all()
    assert convertido.filter(like='nominal__').isin([0, 1]).all().all()
    np.testing.assert_array_equal(convertido['contrato__Contract'], original['Contract'].map(dict(zip(orden_contrato, [0, 1, 2]))))
assert X_train_transformado.columns.equals(X_test_transformado.columns)
print('Verificaciones correctas: ajuste solo en train, escalas esperadas y etiquetas alineadas.')

Verificaciones correctas: ajuste solo en train, escalas esperadas y etiquetas alineadas.


## 5. Exportación para modelado

Exportamos las matrices StandardScaler y las etiquetas originales a las rutas finales acordadas. Volvemos a leerlas para comprobar valores, nombres y orden. El índice `fila_origen` debe leerse con `index_col='fila_origen'` y nunca utilizarse como predictor. Los CSV no guardan el preprocesador: para transformar nuevos clientes, ejecutar este notebook o conservar el objeto ajustado al integrar el pipeline; no volver a ajustarlo sobre test.

In [5]:
salida = RAIZ / 'data' / 'processed'
salida.mkdir(parents=True, exist_ok=True)
for nombre, datos in [('X_train', X_train_transformado), ('X_test', X_test_transformado), ('y_train', y_train), ('y_test', y_test)]:
    ruta = salida / f'{nombre}.csv'
    datos.to_csv(ruta, index=True, index_label='fila_origen')
    recuperado = pd.read_csv(ruta, index_col='fila_origen')
    esperado = datos.to_frame() if isinstance(datos, pd.Series) else datos
    pd.testing.assert_frame_equal(recuperado, esperado, check_dtype=False, rtol=1e-12, atol=1e-12)
print('Cuatro CSV finales exportados y verificados en:', salida)

Cuatro CSV finales exportados y verificados en: D:\src\tpi-analisis-de-datos\data\processed


**Resultado:** codificación y escalado integrados, comparación de escaladores documentada y matrices finales listas para clasificación. No se modificaron los CSV de las tareas 21 y 22 ni se ajustaron transformaciones con test. La tarea 24 revisará el flujo completo antes del modelado.